# 第17章 代码教学：RAG（检索增强生成）

> 目标：
> 1) 构建最小 RAG Pipeline（切块 -> 检索 -> 生成）
> 2) 对比无检索 vs 有检索
> 3) 输出命中文档与相似度分数（可解释）


## 0. 环境准备

依赖：`scikit-learn` `transformers` `torch`。


In [1]:
# !pip install -U scikit-learn transformers accelerate
import numpy as np, torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
np.random.seed(42); torch.manual_seed(42)

device='cuda' if torch.cuda.is_available() else 'cpu'
print('device=', device)


C:\Users\250010108\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device= cpu


## 1. 构建知识库（课程主题片段）


In [2]:
docs = [
    {"id":"doc_attention","text":"Self-attention computes token interactions by query key value projections. Multi-head attention learns diverse subspaces."},
    {"id":"doc_peft","text":"LoRA freezes base weights and trains low-rank matrices. It reduces trainable parameters."},
    {"id":"doc_rlhf","text":"RLHF aligns model outputs with human preference via reward modeling and policy optimization such as PPO or DPO."},
    {"id":"doc_moe","text":"MoE activates only a subset of experts per token. Routing and load balancing are critical."},
    {"id":"doc_rag","text":"RAG retrieves relevant passages before generation, improving factuality and traceability."},
    {"id":"doc_kvcache","text":"KV cache stores previous attention keys and values, reducing autoregressive decoding cost."},
]
print('docs=', len(docs))


docs= 6


## 2. 文本切块（Chunking）

简单的词窗口 + overlap；真实系统可使用更复杂策略。


In [3]:
def chunk_text(text, chunk_size=24, overlap=6):
    w = text.split(); out = []; s = 0
    while s < len(w):
        e = min(s + chunk_size, len(w))
        out.append(' '.join(w[s:e]))
        if e == len(w): break
        s = e - overlap
    return out

chunks = []
for d in docs:
    for i, ch in enumerate(chunk_text(d['text'])):
        chunks.append({'chunk_id': f"{d['id']}_c{i}", 'doc_id': d['id'], 'text': ch})

print('chunks=', len(chunks))
print(chunks[0])


chunks= 6
{'chunk_id': 'doc_attention_c0', 'doc_id': 'doc_attention', 'text': 'Self-attention computes token interactions by query key value projections. Multi-head attention learns diverse subspaces.'}


## 3. 建立检索器（TF-IDF + Cosine）


In [4]:
vec = TfidfVectorizer(ngram_range=(1,2), lowercase=True)
X = vec.fit_transform([c['text'] for c in chunks])

def retrieve(q, top_k=3):
    qv = vec.transform([q])
    sims = cosine_similarity(qv, X)[0]
    ids = np.argsort(sims)[::-1][:top_k]
    out = []
    for i in ids:
        item = dict(chunks[i]); item['score'] = float(sims[i]); out.append(item)
    return out

for h in retrieve('How does LoRA reduce parameters?', 3):
    print(h['doc_id'], h['score'])


doc_peft 0.2871094019135437
doc_kvcache 0.0
doc_rag 0.0


## 4. 加载生成模型


In [5]:
GEN = 'google/flan-t5-small'
# 备选：'t5-small'

tok = AutoTokenizer.from_pretrained(GEN)
model = AutoModelForSeq2SeqLM.from_pretrained(GEN).to(device)
model.eval(); print('loaded', GEN)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights:   1%|          | 1/190 [00:00<00:00, 23967.45it/s, Materializing param=decoder.block.0.layer.0.SelfAttention.k.weight]

Loading weights:   1%|          | 1/190 [00:00<00:00, 2068.20it/s, Materializing param=decoder.block.0.layer.0.SelfAttention.k.weight] 

Loading weights:   1%|          | 2/190 [00:00<00:00, 1802.84it/s, Materializing param=decoder.block.0.layer.0.SelfAttention.o.weight]

Loading weights:   1%|          | 2/190 [00:00<00:00, 1265.06it/s, Materializing param=decoder.block.0.layer.0.SelfAttention.o.weight]

...

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2064.58it/s, Materializing param=shared.weight] 

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2061.59it/s, Materializing param=shared.weight]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2057.12it/s, Materializing param=shared.weight]


The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


loaded google/flan-t5-small


## 5. RAG 生成函数


In [6]:
def rag_prompt(q, hits):
    ctx = "\n".join([f"[{i+1}] {h['text']}" for i, h in enumerate(hits)])
    return (
        "Answer with retrieved evidence. If insufficient, say insufficient evidence.\n"
        f"Question: {q}\nContext:\n{ctx}\nAnswer:"
    )

@torch.no_grad()
def generate(prompt, max_new_tokens=96):
    x = tok(prompt, return_tensors='pt', truncation=True).to(device)
    y = model.generate(**x, max_new_tokens=max_new_tokens, num_beams=4)
    return tok.decode(y[0], skip_special_tokens=True).strip()

def rag_answer(q, top_k=3):
    hits = retrieve(q, top_k)
    ans = generate(rag_prompt(q, hits))
    return {'q': q, 'ans': ans, 'hits': hits}


## 6. 基线：无检索直接回答


In [7]:
def direct_answer(q):
    return generate(f"Answer briefly.\nQuestion: {q}\nAnswer:")

q = 'Why is retrieval useful in RAG?'
print('direct=', direct_answer(q))
out = rag_answer(q, 3)
print('rag=', out['ans'])
for h in out['hits']: print(h['doc_id'], h['score'])


direct= retrieval is useful in RAG.
rag= improving factuality and traceability
doc_rag 0.23400230699614644
doc_kvcache 0.0
doc_moe 0.0


## 7. 小评测集：Direct vs RAG


In [8]:
eval_set = [
    {'q':'What does LoRA train?','gold':'doc_peft'},
    {'q':'What is stored in KV cache?','gold':'doc_kvcache'},
    {'q':'Why is load balancing important in MoE?','gold':'doc_moe'},
    {'q':'How does RAG improve factuality?','gold':'doc_rag'},
]
rows = []
for x in eval_set:
    d = direct_answer(x['q'])
    r = rag_answer(x['q'], 3)
    rows.append({'q': x['q'], 'direct': d, 'rag': r['ans'], 'top_doc': r['hits'][0]['doc_id'], 'gold': x['gold']})

for r in rows:
    print('Q=', r['q'])
    print('Direct=', r['direct'])
    print('RAG=', r['rag'])
    print('TopDoc=', r['top_doc'], 'Gold=', r['gold'])
    print('-'*60)


Q= What does LoRA train?
Direct= a psychiatric unit
RAG= low-rank matrices
TopDoc= doc_peft Gold= doc_peft
------------------------------------------------------------
Q= What is stored in KV cache?
Direct= archival data
RAG= previous attention keys and values
TopDoc= doc_kvcache Gold= doc_kvcache
------------------------------------------------------------
Q= Why is load balancing important in MoE?
Direct= balancing is a key component of balancing.
RAG= RAG retrieves relevant passages before generation, improving factuality and traceability
TopDoc= doc_moe Gold= doc_moe
------------------------------------------------------------
Q= How does RAG improve factuality?
Direct= RAG is a non-profit organization based in Washington, D.C.
RAG= [1]
TopDoc= doc_rag Gold= doc_rag
------------------------------------------------------------


## 8. 检索指标：Hit@k


In [9]:
def hit_at_k(eval_items, k=3):
    hit = 0
    for x in eval_items:
        doc_ids = [h['doc_id'] for h in retrieve(x['q'], k)]
        if x['gold'] in doc_ids: hit += 1
    return hit / len(eval_items)

for k in [1,2,3,4]:
    print(f'Hit@{k}=', hit_at_k(eval_set, k))


Hit@1= 1.0
Hit@2= 1.0
Hit@3= 1.0
Hit@4= 1.0


## 9. 证据展示（可解释）


In [10]:
demo = rag_answer('Explain KV cache in one sentence.', 3)
print('answer=', demo['ans'])
for i, h in enumerate(demo['hits'], 1):
    print(f"[{i}]", h['doc_id'], 'score=', h['score'])
    print('   ', h['text'])


answer= [1]
[1] doc_kvcache score= 0.36982202883525805
    KV cache stores previous attention keys and values, reducing autoregressive decoding cost.
[2] doc_rag score= 0.0
    RAG retrieves relevant passages before generation, improving factuality and traceability.
[3] doc_moe score= 0.0
    MoE activates only a subset of experts per token. Routing and load balancing are critical.


## 10. 练习
1) 换成向量检索；2) 加 query rewrite；3) 在答案中标注引用来源。
